# Notebook 02 — LLM Uncertainty

For each transcript produced by notebook 01, generate a customer-service-style
response with `Qwen/Qwen2.5-3B-Instruct` (4-bit quantized) and extract two
uncertainty signals:

1. **Token entropy** — Shannon entropy of the next-token distribution at each
   generation step (mean and max across the sequence).
2. **Semantic entropy** (simplified Kuhn et al., 2023) — sample N responses,
   cluster on Sentence-BERT cosine distance (threshold 0.3), compute entropy
   over cluster sizes.

**Input**: `output/01_whisper_results.csv` (must exist — run notebook 01 first)
**Output**: `output/02_llm_uncertainty.csv`

**Hardware**: **CUDA recommended** — 4-bit Qwen2.5-3B uses ~3 GB VRAM. Without a
GPU, the notebook falls back to **fp16 on CPU** (slow; ~8–10+ GB RAM for 3B).
Restart this kernel before running notebook 03 to free VRAM after GPU runs.


## 1. Imports and model precision

On **CUDA**: `BitsAndBytesConfig(load_in_4bit=True)` (NF4 + fp16 compute) fits
Qwen2.5-3B into ~3 GB VRAM. On **CPU only**: 4-bit is unavailable — we skip
`BitsAndBytesConfig` and load the model in **fp16** on CPU instead.


In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from scipy.stats import entropy as scipy_entropy
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = REPO_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

use_cuda = torch.cuda.is_available()
device = "cuda" if use_cuda else "cpu"
if use_cuda:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
else:
    bnb_config = None
    print(
        "CUDA not available — loading Qwen in fp16 on CPU (4-bit is GPU-only). "
        "Expect long runtimes; ensure enough RAM (~8–10+ GB for 3B fp16)."
    )


C:\GIT\dialog-uncertainty-toolkit\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: This notebook requires CUDA. 4-bit BitsAndBytes is GPU-only.

## 2. Load Qwen2.5-3B-Instruct

First call downloads ~6 GB to `~/.cache/huggingface/`. `device_map="auto"`
places the quantized weights on GPU.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if use_cuda:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map=None,
        low_cpu_mem_usage=True,
    )
    model = model.to(device)
model.eval()


## 3. `generate_with_entropy` — token-level entropy

Wrap each transcript in a customer-service system prompt, then sample one
response with `temperature=0.7`. For every generation step we compute Shannon
entropy of the next-token distribution and report mean and max across the
generated sequence.


In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful customer service assistant. "
    "Answer the user's question concisely, politely, and accurately. "
    "If you don't know the answer, say so."
)


def generate_with_entropy(
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 0.7,
) -> dict[str, Any]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True,
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[-1]
    generated_ids = out.sequences[0][prompt_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    token_entropies: list[float] = []
    for logits in out.scores:
        probs = torch.softmax(logits[0].float(), dim=-1)
        ent = -(probs * torch.log(probs + 1e-12)).sum().item()
        token_entropies.append(ent)

    return {
        "text": text,
        "mean_token_entropy": float(np.mean(token_entropies)),
        "max_token_entropy": float(np.max(token_entropies)),
    }


## 4. `semantic_uncertainty` — simplified Kuhn et al. (2023)

Sample N responses, embed with `all-MiniLM-L6-v2`, agglomeratively cluster on
cosine **distance** with `distance_threshold=0.3` (i.e., responses with cosine
similarity ≥ 0.7 are treated as semantically equivalent), then take Shannon
entropy over the cluster size distribution. Higher value → the model produces
semantically diverse answers → higher semantic uncertainty.

This is a simplification of the original NLI-based clustering; see
`docs/methodology.md`.


In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)


def semantic_uncertainty(
    prompt: str,
    n_samples: int = 5,
    distance_threshold: float = 0.3,
) -> dict[str, Any]:
    responses = [generate_with_entropy(prompt)["text"] for _ in range(n_samples)]

    embs = embedder.encode(responses, normalize_embeddings=True)
    cosine_distance = 1.0 - (embs @ embs.T)
    np.fill_diagonal(cosine_distance, 0.0)
    cosine_distance = np.clip(cosine_distance, 0.0, None)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric="precomputed",
        linkage="average",
    ).fit(cosine_distance)

    cluster_counts = np.bincount(clustering.labels_)
    cluster_probs = cluster_counts / cluster_counts.sum()
    sem_ent = scipy_entropy(cluster_probs)

    return {
        "semantic_entropy": float(sem_ent),
        "n_clusters": int(len(cluster_counts)),
        "responses": responses,
    }


## 5. Run pipeline over transcripts from notebook 01

Load `output/01_whisper_results.csv`, treat each `transcript` as a customer
query, and compute both uncertainty signals per transcript.

For 20 transcripts × `n_samples=5` semantic samples + 1 single generation each
= ~120 generations total. Expect ~5–10 minutes on RTX 3060.


In [ ]:
whisper_csv = OUTPUT_DIR / "01_whisper_results.csv"
if not whisper_csv.exists():
    raise FileNotFoundError(
        f"{whisper_csv} not found. Run notebook 01 first."
    )

whisper_df = pd.read_csv(whisper_csv)

records: list[dict[str, Any]] = []
for _, row in whisper_df.iterrows():
    transcript = str(row["transcript"]).strip()
    if not transcript:
        continue
    single = generate_with_entropy(transcript)
    sem = semantic_uncertainty(transcript, n_samples=5)
    records.append({
        "file": row["file"],
        "transcript": transcript,
        "response": single["text"],
        "mean_token_entropy": single["mean_token_entropy"],
        "max_token_entropy": single["max_token_entropy"],
        "semantic_entropy": sem["semantic_entropy"],
        "n_clusters": sem["n_clusters"],
    })


## 6. Save results and sanity-check

Write to `output/02_llm_uncertainty.csv` and preview the first rows. Expected
ranges (rough): `mean_token_entropy` ~0.5–2.5 nats, `semantic_entropy` 0–1.6
nats (capped at `ln(n_samples) ≈ 1.61` for n=5).


In [ ]:
df = pd.DataFrame(records)
csv_path = OUTPUT_DIR / "02_llm_uncertainty.csv"
df.to_csv(csv_path, index=False)

print(f"Saved {len(df)} rows to {csv_path.relative_to(REPO_ROOT)}")
df.head(10)
